In [ ]:
import pandas as pd
import re
from pathlib import Path
import os
import pandas as pd
import numpy as np
import torch
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics.pairwise import cosine_similarity
from transformers import AutoTokenizer, AutoModel
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

In [ ]:
def run_ladder_validation(df,
                          out_dir="Intermediate Files",
                          csv_name="Semantic_results_general_only.csv",
                          max_length=512,
                          device=None,
                          run_name=None,
                          add_timestamp=False):

    import os
    import numpy as np
    import pandas as pd
    import torch
    from tqdm import tqdm
    from transformers import AutoTokenizer, AutoModel
    from sklearn.metrics.pairwise import cosine_similarity

    MODELS = {
        'BioLORD-2023': 'FremyCompany/BioLORD-2023',
        'MedCPT': 'ncbi/MedCPT-Query-Encoder'
    }

    METHOD_NAMES = {
        'Our': 'LADDER',
        'Hu': 'Hu et al',
        'GeneAgent': 'GeneAgent'
    }

    if device is None:
        device = "cuda" if torch.cuda.is_available() else "cpu"

    os.makedirs(out_dir, exist_ok=True)

    prefix = f"{run_name}_" if run_name else ""
    if add_timestamp:
        from datetime import datetime
        ts = datetime.now().strftime("%Y%m%dT%H%M%S")
        prefix = f"{prefix}{ts}_" if prefix else f"{ts}_"

    csv_final = os.path.join(out_dir, f"{prefix}{csv_name}" if prefix else csv_name)

    def load_model(model_id):
        tokenizer = AutoTokenizer.from_pretrained(model_id)
        model = AutoModel.from_pretrained(model_id)
        model.to(device)
        model.eval()
        return tokenizer, model

    def get_embedding(text, tokenizer, model):
        if not isinstance(text, str) or len(text.strip()) == 0:
            return np.zeros(model.config.hidden_size, dtype=float)

        inputs = tokenizer(text,
                           return_tensors="pt",
                           truncation=True,
                           max_length=max_length,
                           padding=True)

        inputs = {k: v.to(device) for k, v in inputs.items()}

        with torch.no_grad():
            outputs = model(**inputs)
            emb = outputs.last_hidden_state[:, 0, :].cpu().numpy().flatten()

        return emb

    def validate_with_model(df_local, display_name, model_id):
        tokenizer, model = load_model(model_id)

        rows = []

        for _, row in tqdm(df_local.iterrows(), total=len(df_local), desc=display_name):

            desc_emb = get_embedding(row.get('MSigDB_Brief_Description', ''), tokenizer, model)
            hu_emb = get_embedding(row.get('Hu Annotation', ''), tokenizer, model)
            ga_emb = get_embedding(row.get('GeneAgent Annotation', ''), tokenizer, model)
            our_emb = get_embedding(row.get('Our Annotation', ''), tokenizer, model)

            eps = 1e-12

            hu_sim = cosine_similarity(desc_emb.reshape(1, -1)+eps,
                                       hu_emb.reshape(1, -1)+eps)[0][0]

            ga_sim = cosine_similarity(desc_emb.reshape(1, -1)+eps,
                                       ga_emb.reshape(1, -1)+eps)[0][0]

            our_sim = cosine_similarity(desc_emb.reshape(1, -1)+eps,
                                        our_emb.reshape(1, -1)+eps)[0][0]

            scores = {'Our': our_sim, 'Hu': hu_sim, 'GeneAgent': ga_sim}
            winner = max(scores, key=scores.get)

            rows.append({
                "Geneset": row.get("Geneset"),
                "LADDER_Similarity": our_sim,
                "Hu_Similarity": hu_sim,
                "GeneAgent_Similarity": ga_sim,
                "Winner": METHOD_NAMES[winner],
                "Model": display_name
            })

        del model, tokenizer
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

        return pd.DataFrame(rows)

    results = []

    for name, path in MODELS.items():
        try:
            res = validate_with_model(df, name, path)
            results.append(res)
        except Exception as e:
            print(f"Model failed: {name} -> {e}")

    results_df = pd.concat(results, ignore_index=True)

    results_df.to_csv(csv_final, index=False)

    print(f"Saved results: {csv_final}")

    return results_df

In [ ]:
combined_path = "/Users/justin.seby/Documents/venv/Justin/Karolinska Institutet/DDLS/DDLS Code/DDLS Projects/LADDER Code Repo/Ladder Annotation and Validation/Benchmarking/AML/AMLWithoutContext_Combined_Annotations.csv"

combined = pd.read_csv(combined_path)

combined_first6 = combined.iloc[:, :5]

annotations_df = combined_first6

msigdb_df = pd.read_csv('AML_msigdb_descriptions.csv')

df = annotations_df.merge(msigdb_df, on='Geneset')

df["Our Annotation"] = (
    df["Our Annotation"]
    .str.replace(r"\bacute myeloid leukemia\b", "", case=False, regex=True)
    .str.replace(r"\baml\b", "", case=False, regex=True)
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)


print(f"Total genesets: {len(df)}\n")


results_df = run_ladder_validation(
    df,
    run_name="AMLWITHOUTCONTEXTMASKED",
    add_timestamp=False
)